In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import BaggingRegressor, ExtraTreesRegressor, RandomForestRegressor, GradientBoostingRegressor, StackingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR 
from matplotlib.pyplot import MultipleLocator
from sklearn.model_selection import GridSearchCV, train_test_split
from catboost import CatBoostRegressor
import lightgbm as lgb
import pandas as pd
import numpy as np
from math import sqrt
from statistics import mean
from scipy import stats
from pytorch_tabnet.tab_model import TabNetRegressor
from scipy.stats import pearsonr
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
from matplotlib.font_manager import FontProperties, fontManager
from sklearn.utils import parallel_backend

module_path = ''
import sys
if module_path not in sys.path:
    sys.path.append(module_path)

from Transformer import *

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"
def regression_analysis(X, X_test1, y, y_test1,dataset_name,model_code,work):
    base_path = ''
    pict_path = base_path + dataset_name + '_'
    if model_code == 'polynomial':
        poly = PolynomialFeatures(degree=3)
        poly.fit(X)
        X = poly.transform(X)
        X_test1 = poly.transform(X_test1)
        model = LinearRegression()
        param_grid = {}
        pict_path += 'PR.png'
    elif model_code == 'bagging':
        model = BaggingRegressor()
        param_grid = {'n_estimators': [1000,1500], 'max_features': [3,4,5], 'max_samples': [0.7,0.8,0.9]}
        pict_path += 'BAGGING.png'
    elif model_code == 'extratrees':
        model = ExtraTreesRegressor()
        param_grid = {'n_estimators': [1000,1500], 'max_features': [3,4,5], 'max_depth': [7,9]}
        pict_path += 'ET.png'
    elif model_code == 'randomforest':
        model = RandomForestRegressor()
        param_grid = {'n_estimators': [6000,4500,7500], 'max_features': [3,4,5], 'max_depth': [7,9]}
        pict_path += 'RF.png'
    elif model_code == 'gradientboosting':
        model = GradientBoostingRegressor()
        param_grid = {'learning_rate': [0.1,0.01,0.05], 'n_estimators': [1000], 'max_depth': [7,9]}
        pict_path += 'GBDT.png'
    elif model_code == 'kneighbors':
        model = KNeighborsRegressor()
        param_grid = {'n_neighbors': [25,30,35]}
        pict_path += 'KNN.png'
    elif model_code == 'mlpregressor':
        model = MLPRegressor()
        param_grid = {'hidden_layer_sizes': [(7, 100, 7)]}
        pict_path += 'ANN.png'
    elif model_code == 'lgbm':
        model = lgb.LGBMRegressor()
        param_grid = {'n_estimators': [300,400,500], 'learning_rate': [0.01,0.05,0.1], 'max_depth': [7,9]}
        pict_path += ':LGBM.png'
    elif model_code == 'stacking1':
        param_grid = {
            'stacking__GradientBoostingRegressor__learning_rate': [0.015],
            'stacking__GradientBoostingRegressor__n_estimators': [1800],
            'stacking__GradientBoostingRegressor__max_depth': [9],
            'stacking__transformer__lr': [0.01,0.001,0.005,0.0005,0.0001,0.00005],#0.01,0.001,0.005,
            'stacking__transformer__batch_size': [256]
              
        }
        estimators = [('GradientBoostingRegressor', GradientBoostingRegressor()),
                      ('transformer', transformer(in_features=X.shape[1]))]
        final_estimator = LinearRegression()
        model = StackingRegressor(estimators=estimators, final_estimator=final_estimator, n_jobs=-1)
        pict_path += 'stacking_GBDT.png'
    elif model_code == 'TabNet':   
        # n_estimators = np.arange(300,600,20)
        model = TabNetRegressor()
        pict_path += 'TabNet.png'
    elif model_code == 'SVR':
        model = SVR()
        param_grid = { 'degree':[2],
                'C': [10],
                'epsilon': [1],
                'gamma':[1]}
        pict_path += 'SVR.png'
    elif model_code == 'stacking2':
        param_grid = {
            'stacking__ExtraTreesRegressor__max_features': [3,5],
            'stacking__ExtraTreesRegressor__n_estimators': [1500],
            'stacking__ExtraTreesRegressor__max_depth': [9]
        }
        estimators = [('ExtraTreesRegressor', ExtraTreesRegressor()),
                      ('transformer', transformer(in_features=X.shape[1]))]
        final_estimator = LinearRegression()
        model = StackingRegressor(estimators=estimators, final_estimator=final_estimator, n_jobs=-1)
        pict_path += 'stacking_ET.png'
    elif model_code == 'stacking3':
        param_grid = {
            'stacking__CatBoostRegressor__learning_rate': [0.025],
            'stacking__CatBoostRegressor__iterations': [1200],
            'stacking__CatBoostRegressor__depth': [7]
        }
        estimators = [('CatBoostRegressor', CatBoostRegressor()),
                      ('transformer', transformer(in_features=X.shape[1]))]
        final_estimator = LinearRegression()
        model = StackingRegressor(estimators=estimators, final_estimator=final_estimator, n_jobs=-1)
        pict_path += 'stacking_Cat.png' 

    else:
        print("Invalid model code.")
        return
    



    if work == 'sandian':
        if model_code == 'TabNet':
            X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=1)
            y_train=y_train.reshape(-1,1)
            y_test=y_test.reshape(-1,1)
            with parallel_backend('threading'):
                grid_search = model
                grid_search.fit(X_train, y_train)
        elif model_code == 'stacking1'or'stacking2'or'stacking3':
            pipeline = Pipeline([
                ('stacking', model)
                ])
            X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=1)
            grid_search = GridSearchCV(pipeline, param_grid)
            grid_search.fit(X_train, y_train)
        else:
            grid_search = GridSearchCV(model, param_grid, cv=5, n_jobs=-1)
            X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=1)
            grid_search.fit(X_train, y_train)
        

        pred=grid_search.predict(X_test1)
        if model_code != 'TabNet':
            print("R2: {:.6f}".format(grid_search.score(X_test, y_test)))
            print("Best parameters: {}".format(grid_search.best_params_))   
            print("Best cross-validation score: {:.3f}".format(grid_search.best_score_))
            results = pd.DataFrame(grid_search.cv_results_)
            display(results.head())
        print("校正后")
        if  model_code != 'TabNet':
            error = y_test1 - pred
        else :
            error = y_test1 - pred[:,0]  
        squaredError = np.square(error)
        absError = np.abs(error)
        MAE = np.mean(absError)  # 平均绝对误差MAE
        RMSE = sqrt(np.mean(squaredError))  # 均方根误差RMSE
        SD = np.std(error)  # 标准差
        print("MAE = {:.6f}".format(np.mean(absError)))
        print("RMSE = {:.6f}".format(sqrt(np.mean(squaredError))))
        print("SD = {:.6f}".format(np.std(error)))
        sorted_height_diff = np.sort(np.abs(error))
        le90_index = int(0.9 * len(sorted_height_diff))
        le90 = sorted_height_diff[le90_index]
        print("LE90 = {:.6f}".format(le90))

        x_san=np.squeeze(pred)
        y_san=np.array(y_test1) 

        # 计算散点密度
        xy_san = np.vstack([x_san, y_san])
        z = stats.gaussian_kde(xy_san)(xy_san)
        idx_san = z.argsort()
        x_san, y_san, z = x_san[idx_san], y_san[idx_san], z[idx_san] 
        # 拟合（若换MK，自行操作）最小二乘
        def slope(x_sans, y_sans):
            m = (((mean(x_sans) * mean(y_sans)) - mean(x_sans * y_sans)) / ((mean(x_sans) * mean(x_sans)) - mean(x_sans * x_sans)))
            b = mean(y_sans) - m * mean(x_sans)
            return m, b
        k, b = slope(y_san, x_san)
        regression_line = []
        for a in y_san:
            regression_line.append((k * a) + b)
        R2 = pearsonr(x_san, y_san)[0]
        
        
        # 指定Times New Roman字体文件的路径
        font_path = r''
        # 加载这个字体
        times_new_roman_font = FontProperties(fname=font_path).get_name()
        # 将这个字体添加到Matplotlib的字体列表中
        fontManager.addfont(font_path)
        config = {"font.family":'times_new_roman_font',"font.size": 16,"mathtext.fontset":'stix'}
        fig, ax = plt.subplots(figsize=(4, 3), dpi=300)
        scatter = ax.scatter(y_san, x_san, marker='o', c=z * 1, edgecolors=None, s=5, cmap='RdBu_r',  alpha=0.8)
        cbar = plt.colorbar(scatter, shrink=1, orientation='vertical', extend='both', pad=0.015, aspect=30, label='frequency')
        plt.plot([-30, 20], [-30, 20], 'red', lw=1.5, linestyle='--', label='1:1 line') 
        plt.plot(y_san, regression_line, 'black', lw=1.5, label='Regression Line')  
        ax.grid(True, linestyle='--', alpha=0.2)

        plt.text(19,-19, '$R^2=%.3f$' % R2, family = 'Times New Roman', horizontalalignment='right')
        plt.text(19,-22, '$MAE=%.3f$' % MAE, family = 'Times New Roman', horizontalalignment='right')
        plt.text(19,-25, '$RMSE=%.3f$' % RMSE, family = 'Times New Roman', horizontalalignment='right')
        plt.text(19,-28, '$SD=%.3f$' % SD, family = 'Times New Roman', horizontalalignment='right')

        plt.axis([-30, 20, -30, 20])  # 设置线的范围
        ax.legend(loc='upper left', frameon = False)
        plt.xlabel('Predicted height error (m)',family = 'Times New Roman')
        plt.ylabel('True height error (m)',family = 'Times New Roman')
        plt.savefig(pict_path, dpi=700, bbox_inches='tight')
        plt.show()
        print("R2 = {:.3f}".format(R2))
        print("MAE =  {:.3f}".format(MAE))#平均绝对误差MAE
        print("RMSE = {:.3f}".format(RMSE))#均方根误差RMSE
        print("SD = {:.3f}".format(SD))#标准差
        print("LE90 = {:.3f}".format(le90))
    
        pict_path = base_path + dataset_name + '_' +'zhexian.png'
        X_all = np.vstack((X, X_test1))
        y_all = np.hstack((y, y_test1))
        pred_all = grid_search.predict(X_all)
        error=(pred_all-y_all)
        aux=np.where((abs(error)<15))
        error=error[aux]
        
        with plt.style.context({'figure.figsize': (30, 8), 'font.size': 28}):
            plt.figure()
            plt.plot(error, c='r')
            plt.tick_params(labelsize=30)
            ax = plt.gca()
            y_major_locator = MultipleLocator(10)
            ax.yaxis.set_major_locator(y_major_locator)
            plt.xlabel("Sample points", fontsize=40)
            plt.ylabel("Absolute error", fontsize=40)
            plt.ylim(-20, 20)
            plt.xlim(-100, X_all.shape[0])
            plt.savefig(pict_path, dpi=700, bbox_inches='tight')
            plt.show()
    elif work == 'moxing':
        from sklearn.model_selection import ParameterGrid
        import joblib
        param_grid = ParameterGrid(param_grid)
        base_path1 = '/home/yucl/project/ZJH/moxing/'
        pipeline = Pipeline([
        ('stacking', model)
        ])
        X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=1)
        for i, params in enumerate(param_grid):
            learning_rate = params['stacking__transformer__lr']
            batch_size = params['stacking__transformer__batch_size']
            
            # 设置当前参数组合
            pipeline.set_params(**params)
            
            # 构建文件路径
            file_name = f'_lr_{learning_rate}_size_{batch_size}_stacking.pkl'
            path = base_path1 + dataset_name + file_name
            
            # 检查模型文件是否已经存在
            if not os.path.exists(path):
                # 拟合模型
                pipeline.fit(X_train, y_train)
                
                # 如果文件不存在，则保存模型
                joblib.dump(pipeline, path)
                print(f'模型参数 {params} 已保存到 {path}')
            else:
                print(f'模型参数 {params} 已存在，跳过...')
        
        # joblib.dump(grid_search, path)


# 使用示例
# regression_analysis(X, X_test1, y, y_test1, 'polynomial')


In [ ]:
xlsx1 = pd.read_excel(r'' , engine='openpyxl', sheet_name=0)
X_FIR=xlsx1.values[:,0:7]
y_FIR=xlsx1.values[:,8]

transfer = MinMaxScaler(feature_range=(0, 1))
    # 2. 调用fit_transform （只需要处理特征）
X1_FIR = transfer.fit_transform(X_FIR)
X, X_test1, y, y_test1 = train_test_split(X1_FIR, y_FIR, random_state=52)

regression_analysis(X, X_test1, y, y_test1,'','','')

